# Digital Inspector — family & stage classifier training

Trains the two mmBERT-small classifiers from CLAUDE.md §5, calibrates the family model, exports both to ONNX INT8, and builds the FAISS similarity index over `scripts_index.jsonl`.

**Kaggle setup before running:**
1. Settings → Accelerator → GPU (T4 x2 or P100).
2. Settings → Internet → On (needed to pull base models from the Hub).
3. Add Data → upload `train.jsonl`, `eval.jsonl`, `stage_labels.jsonl`, `scripts_index.jsonl` from `data/processed/` as a Kaggle Dataset, or point `DATA_DIR` below at wherever you've attached them.

Outputs land in `/kaggle/working/models/` — Kaggle keeps this when you "Save Version", which is this notebook's equivalent of "checkpoint to Drive, not local disk".

In [ ]:
!pip install "optimum[onnxruntime]" onnx onnxruntime

In [4]:
!pip install -q -U "transformers>=4.48"
!pip install -q faiss-cpu sentencepiece accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 108.1 MB/s eta 0:00:0000:0100:01


In [41]:
import os
from pathlib import Path

print("cwd:", os.getcwd())
print("files in /content:", [p.name for p in Path("/content").glob("*.jsonl")])
print("files in /content/models:", [p.name for p in Path("/content/models").glob("*")])
print("train exists:", Path("/content/train.jsonl").exists())
print("eval exists:", Path("/content/eval.jsonl").exists())
print("stage exists:", Path("/content/stage_labels.jsonl").exists())
print("scripts exists:", Path("/content/scripts_index.jsonl").exists())

cwd: /content
files in /content: ['stage_labels.jsonl', 'eval.jsonl', 'train.jsonl', 'scripts_index.jsonl']
files in /content/models: ['config.json', 'tokenizer.json', 'special_tokens_map.json', 'tokenizer_config.json']
train exists: True
eval exists: True
stage exists: True
scripts exists: True


In [1]:
!nvidia-smi

Tue Jul 14 10:45:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [24]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import subprocess
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset

SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA_DIR_CANDIDATES = [
    Path.cwd() / "data/processed",
    Path.cwd(),
    Path("/content/drive/MyDrive/digital-inspector/data/processed"),
    Path("/content/drive/MyDrive/data/processed"),
    Path("/content/digital-inspector/data/processed"),
    Path("/content") / "data/processed",
    Path("/content"),
    Path("/content/sample_data"),
    Path("/kaggle/input/digital-inspector/data/processed"),
    Path("../data/processed"),
    Path("data/processed"),
]
REQUIRED_FILES = ["train.jsonl", "eval.jsonl", "stage_labels.jsonl", "scripts_index.jsonl"]
DATA_DIR = None
for candidate in DATA_DIR_CANDIDATES:
    if all((candidate / name).exists() for name in REQUIRED_FILES):
        DATA_DIR = candidate
        break
if DATA_DIR is None:
    searched = [str(path) for path in DATA_DIR_CANDIDATES]
    raise FileNotFoundError(f"Could not find the processed data folder. Upload the four JSONL files into one folder and set DATA_DIR to it. Searched: {searched}")

OUTPUT_DIR = Path("/kaggle/working/models")
if not Path("/kaggle/working").exists():
    OUTPUT_DIR = Path("models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR={DATA_DIR} (exists={DATA_DIR.exists()})")
print(f"OUTPUT_DIR={OUTPUT_DIR}")

DATA_DIR=/content (exists=True)
OUTPUT_DIR=models


## Base model canary

`jhu-clsp/mmBERT-small` needs `transformers>=4.48` for ModernBERT architecture support. Verify it loads *before* anything else — if it fails, fall back immediately rather than debugging the base model choice.

In [38]:
from transformers import AutoModel, AutoTokenizer, __version__ as tf_version

print(f"transformers version: {tf_version}")

BASE_MODEL_CANDIDATES = [
    Path("/content/models/mmBERT-small"),
    Path("/content/models"),
    Path("/kaggle/input/mmbert-small"),
]
WEIGHT_FILES = ("model.safetensors", "pytorch_model.bin")


def has_weights(path):
    return any((path / w).exists() for w in WEIGHT_FILES)


BASE_MODEL_PATH = next(
    (p for p in BASE_MODEL_CANDIDATES if (p / "config.json").exists() and has_weights(p)),
    None,
)
if BASE_MODEL_PATH is None:
    for p in BASE_MODEL_CANDIDATES:
        if (p / "config.json").exists():
            print(f"{p} has config.json but NO weight file — ignoring it")
    BASE_MODEL = "jhu-clsp/mmBERT-small"
    print(f"falling back to the hub: {BASE_MODEL}")
else:
    BASE_MODEL = str(BASE_MODEL_PATH)
    print(f"using local base model (config + weights present): {BASE_MODEL}")

probe, load_info = AutoModel.from_pretrained(
    BASE_MODEL, reference_compile=False, output_loading_info=True
)
missing = load_info.get("missing_keys", [])
encoder_missing = [k for k in missing if not k.startswith(("classifier", "head", "pooler"))]

print(f"randomly-initialised tensors: {len(missing)}")
if encoder_missing:
    for k in encoder_missing[:8]:
        print(f"   ENCODER NOT LOADED: {k}")
    raise SystemExit(
        "The encoder is randomly initialised — this would train mmBERT FROM SCRATCH, "
        "which is why a transformer can lose to TF-IDF. Fix BASE_MODEL."
    )

probe.eval()
probe_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
probe_texts = [
    "This is Inspector Kumar from Mumbai Police, a warrant has been issued against you",
    "CBI officer here, you are under arrest for money laundering",
    "I would like to order a large pizza with extra cheese and olives",
]
enc = probe_tokenizer(probe_texts, padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    hidden = probe(**enc).last_hidden_state
mask = enc["attention_mask"].unsqueeze(-1).float()
vecs = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
vecs = torch.nn.functional.normalize(vecs, dim=1)
related = float(vecs[0] @ vecs[1])
unrelated = float(vecs[0] @ vecs[2])

print(f"cosine(two police-scam lines): {related:.4f}")
print(f"cosine(police vs pizza order): {unrelated:.4f}")
print(f"separation: {related - unrelated:.4f}")

if related > unrelated:
    print()
    print(f"base model pretrained: scam-pair cosine {related:.4f} > off-topic {unrelated:.4f}, 0 random tensors -> safe")
else:
    print()
    print("WARNING: raw embeddings did not separate the scam pair from the off-topic line.")
    print("raw ModernBERT embeddings are anisotropic (all cosines ~0.95) so this probe is weak;")
    print("the reliable signal is above (0 randomly-initialised encoder tensors). proceeding.")
del probe

transformers version: 5.13.1
Using local base model folder: /content/models


## Load the frozen split

`train.jsonl` / `eval.jsonl` are the family-classifier split from `data/05_split.py` — eval is real/real-derived call-shaped content only (email-sourced `fredzhang7` rows are excluded from eval and capped in train, see `data/README.md`).

In [ ]:
FAMILY_IDS = [
    "digital_arrest",
    "kyc_bank_fraud",
    "parcel_courier",
    "tech_support",
    "refund_reward",
    "investment_fraud",
    "legitimate",
]
FAMILY_LABEL2ID = {label: i for i, label in enumerate(FAMILY_IDS)}
FAMILY_ID2LABEL = {i: label for i, label in enumerate(FAMILY_IDS)}

STAGE_IDS = [
    "s0_none",
    "s1_authority_claim",
    "s2_threat_urgency",
    "s3_isolation",
    "s4_info_harvest",
    "s5_payment_demand",
]
STAGE_LABEL2ID = {label: i for i, label in enumerate(STAGE_IDS)}
STAGE_ID2LABEL = {i: label for i, label in enumerate(STAGE_IDS)}


def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def full_transcript(row):
    return "\n".join(t["text"] for t in row["turns"])


family_train_raw = load_jsonl(DATA_DIR / "train.jsonl")
family_eval_all = load_jsonl(DATA_DIR / "eval.jsonl")
family_eval_raw = [r for r in family_eval_all if r.get("eval_kind", "real_holdout") == "real_holdout"]
family_devonly_raw = [r for r in family_eval_all if r.get("eval_kind") == "dev_only_synthetic_sanity_check"]
print(f"family train: {len(family_train_raw)} rows")
print(f"real-holdout eval: {len(family_eval_raw)} rows (the headline metric)")
print(f"dev-only synthetic sanity rows: {len(family_devonly_raw)} (classes with no real eval, reported separately)")

In [ ]:
from collections import Counter

stage_rows_check = load_jsonl(DATA_DIR / "stage_labels.jsonl")

_train_families = Counter(r["family"] for r in family_train_raw)
_eval_families = Counter(r["family"] for r in family_eval_raw)
_eval_ids = {r["dialogue_id"] for r in family_eval_raw}
_leaked = sum(1 for r in family_train_raw if r.get("parent_dialogue_id") in _eval_ids)
_victim = sum(1 for r in stage_rows_check if r.get("label_method") == "rule_victim_turn")
_da_stage = sum(1 for r in stage_rows_check if r["family"] == "digital_arrest")
_legit_frac = _eval_families["legitimate"] / max(len(family_eval_raw), 1)

problems = []
if _leaked:
    problems.append(f"{_leaked} train rows are augmented variants of eval dialogues (lineage leak)")
if _train_families["parcel_courier"] < 200:
    problems.append(f"parcel_courier has only {_train_families['parcel_courier']} train rows (expected ~248)")
if _victim == 0:
    problems.append("stage_labels.jsonl has no victim turns (03b_add_victim_turns.py never ran)")
if _da_stage == 0:
    problems.append("stage_labels.jsonl has no digital_arrest utterances (03c never ran)")
if _legit_frac > 0.66:
    problems.append(f"eval is {_legit_frac:.1%} legitimate (expected ~62%) — looks like the old SMS-chatter eval")

print(f"train {len(family_train_raw)} | eval {len(family_eval_raw)} | stage {len(stage_rows_check)}")
print(f"parcel_courier train rows : {_train_families['parcel_courier']}")
print(f"eval legitimate fraction  : {_legit_frac:.4f}")
print(f"victim turns in stage data: {_victim}")
print(f"digital_arrest stage rows : {_da_stage}")
print(f"lineage leaks             : {_leaked}")

if problems:
    print()
    for p in problems:
        print(f"  STALE: {p}")
    raise SystemExit(
        "\nThis is the OLD dataset. Re-upload data/processed/{train,eval,stage_labels,scripts_index}.jsonl "
        "to your Kaggle Dataset before training — otherwise you burn a GPU session measuring nothing."
    )

print("\ndata fingerprint OK — this is the fixed corpus")

In [ ]:
import random


def lineage_key(row):
    return row.get("parent_dialogue_id") or row["dialogue_id"]


train_keys = sorted({lineage_key(r) for r in family_train_raw})
rng_split = random.Random(SEED)
rng_split.shuffle(train_keys)
val_keys = set(train_keys[:round(len(train_keys) * 0.12)])

family_fit_raw = [r for r in family_train_raw if lineage_key(r) not in val_keys]
family_val_raw = [r for r in family_train_raw if lineage_key(r) in val_keys]

family_train_ds = Dataset.from_dict({
    "text": [full_transcript(r) for r in family_fit_raw],
    "label": [FAMILY_LABEL2ID[r["family"]] for r in family_fit_raw],
})
family_val_ds = Dataset.from_dict({
    "text": [full_transcript(r) for r in family_val_raw],
    "label": [FAMILY_LABEL2ID[r["family"]] for r in family_val_raw],
})
family_eval_ds = Dataset.from_dict({
    "text": [full_transcript(r) for r in family_eval_raw],
    "label": [FAMILY_LABEL2ID[r["family"]] for r in family_eval_raw],
})

print(f"fit: {len(family_fit_raw)}  val: {len(family_val_raw)}  locked test: {len(family_eval_raw)}")
print("val is carved out of train by augmentation lineage, so a dialogue's synthetic variants")
print("never straddle the fit/val boundary. Early stopping and temperature use val only;")
print("the locked test is touched exactly once, at the end.")

## Run A — family classifier

Full transcript, max_length 2048, 7 labels, class-weighted loss, lr 2e-5, up to 4 epochs with early stop on **validation** macro-F1 (never the locked test).

Median transcript is ~214 tokens against a 2048 ceiling, so padding every row to `max_length` would throw away ~85% of the compute on padding — and attention is quadratic. Training uses dynamic padding (`DataCollatorWithPadding`) plus `group_by_length`, which batches similar-length transcripts together so each batch pads only to its own longest row.

In [28]:
from collections import Counter

family_counts = Counter(family_train_ds["label"])
family_total = sum(family_counts.values())
_raw = np.array([family_total / (len(FAMILY_IDS) * family_counts.get(i, 1)) for i in range(len(FAMILY_IDS))])
_w = _raw
family_class_weights = torch.tensor(_w, dtype=torch.float)
print("family class weights (inverse-frequency, same scheme as the TF-IDF baseline):")
for f, raw, w in zip(FAMILY_IDS, _raw, _w):
    print(f"  {f:18s} {w:6.2f}")
print()
print("parcel_courier now has 248 clean rows rather than 88, so its inverse-frequency")
print("weight falls from 12x to ~4x on its own. The TF-IDF baseline that scores 0.82")
print("macro-F1 uses exactly this scheme (class_weight=balanced).")

family class weights: {'digital_arrest': 2.69960618019104, 'kyc_bank_fraud': 1.4458293914794922, 'parcel_courier': 11.708074569702148, 'tech_support': 1.8539464473724365, 'refund_reward': 0.5809832215309143, 'investment_fraud': 2.919086217880249, 'legitimate': 0.30775511264801025}


In [ ]:
from transformers import AutoTokenizer

family_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
FAMILY_MAX_LENGTH = 2048


def tokenize_family(batch):
    return family_tokenizer(batch["text"], truncation=True, max_length=FAMILY_MAX_LENGTH)


family_train_tok = family_train_ds.map(tokenize_family, batched=True, remove_columns=["text"])
family_val_tok = family_val_ds.map(tokenize_family, batched=True, remove_columns=["text"])
family_eval_tok = family_eval_ds.map(tokenize_family, batched=True, remove_columns=["text"])

_lens = sorted(len(x) for x in family_train_tok["input_ids"])
print(f"token lengths — median {_lens[len(_lens) // 2]}, p95 {_lens[int(len(_lens) * 0.95)]}, max {_lens[-1]}")
print(f"padding to max_length would waste {100 * (1 - sum(_lens) / (len(_lens) * FAMILY_MAX_LENGTH)):.0f}% of compute;")
print("the collator below pads each batch to its own longest row instead")

In [30]:
from transformers import AutoModelForSequenceClassification

family_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(FAMILY_IDS),
    id2label=FAMILY_ID2LABEL,
    label2id=FAMILY_LABEL2ID,
    reference_compile=False,
)

pytorch_model.bin:   0%|          | 0.00/564M [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from transformers import (
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = torch.nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


def make_compute_metrics(id_list):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
            "accuracy": (preds == labels).mean(),
        }
    return compute_metrics

In [ ]:
family_collator = DataCollatorWithPadding(tokenizer=family_tokenizer)

family_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "family_checkpoints"),
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    auto_find_batch_size=True,
    num_train_epochs=6,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    dataloader_num_workers=2,
    logging_steps=25,
    report_to=[],
    seed=SEED,
)

family_trainer = WeightedTrainer(
    model=family_model,
    args=family_args,
    train_dataset=family_train_tok,
    eval_dataset=family_val_tok,
    data_collator=family_collator,
    class_weights=family_class_weights,
    compute_metrics=make_compute_metrics(FAMILY_IDS),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

family_trainer.train()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

ACCURACY_THRESHOLD = 0.80
STRETCH_THRESHOLD = 0.90
TFIDF_BASELINE_ACC = 0.913

family_val_output = family_trainer.predict(family_val_tok)
family_eval_output = family_trainer.predict(family_eval_tok)
family_preds = np.argmax(family_eval_output.predictions, axis=-1)
family_labels = family_eval_output.label_ids

present = sorted(set(family_labels.tolist()))
present_names = [FAMILY_IDS[i] for i in present]
absent_names = [FAMILY_IDS[i] for i in range(len(FAMILY_IDS)) if i not in present]

print(classification_report(family_labels, family_preds, labels=present, target_names=present_names, zero_division=0))
if absent_names:
    print(f"NOTE: {absent_names} have no real-holdout eval rows and are excluded from the headline macro-F1.")
    print("      They are reported below as a synthetic dev-only sanity check.")

family_cm = confusion_matrix(family_labels, family_preds, labels=present)
plt.figure(figsize=(8, 6))
sns.heatmap(family_cm, annot=True, fmt="d", xticklabels=present_names, yticklabels=present_names, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Family classifier — real-holdout call transcripts")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "family_confusion_matrix.png", dpi=150)
plt.show()

LEGIT_ID = FAMILY_LABEL2ID["legitimate"]
true_scam = family_labels != LEGIT_ID
pred_scam = family_preds != LEGIT_ID

family_accuracy = (family_preds == family_labels).mean()
family_macro_f1 = f1_score(family_labels, family_preds, labels=present, average="macro", zero_division=0)
trivial_baseline = (family_labels == LEGIT_ID).mean()
scam_recall = (pred_scam & true_scam).sum() / max(true_scam.sum(), 1)
false_alarm_rate = (pred_scam & ~true_scam).sum() / max((~true_scam).sum(), 1)

print("=" * 62)
print(f"accuracy                        {family_accuracy:.4f}   <- threshold {ACCURACY_THRESHOLD:.2f}")
print(f"macro-F1 (real classes only)    {family_macro_f1:.4f}   <- the honest one")
print(f"always-answer-legitimate        {trivial_baseline:.4f}   <- beat this, not zero")
print()
print(f"scam recall                     {scam_recall:.4f}")
print(f"false-alarm rate on legitimate  {false_alarm_rate:.4f}   <- what a judge probes")
print("=" * 62)

if family_accuracy >= STRETCH_THRESHOLD:
    print(f"PASS+ — family accuracy {family_accuracy:.1%} clears the {STRETCH_THRESHOLD:.0%} stretch bar")
elif family_accuracy >= ACCURACY_THRESHOLD:
    print(f"PASS — family accuracy {family_accuracy:.1%} clears the {ACCURACY_THRESHOLD:.0%} bar")
else:
    print(f"FAIL — family accuracy {family_accuracy:.1%} is under the {ACCURACY_THRESHOLD:.0%} bar")
    print("  Do NOT fix this by touching eval.jsonl. In order:")
    print("   1. generate more call-shaped training data for the weakest family")
    print("   2. raise num_train_epochs / sweep learning_rate")
    print("   3. accept it and report honestly. A real 82% beats a fake 94%.")

if family_accuracy < TFIDF_BASELINE_ACC:
    print()
    print(f"NOTE: a TF-IDF + logistic-regression baseline scores {TFIDF_BASELINE_ACC:.1%} on this split")
    print("      (training/baseline_tfidf.py). The transformer's real edge is Hinglish/Devanagari,")
    print("      which this mostly-English eval barely tests, so parity here is acceptable.")

print()
print("accuracy flatters this eval (legitimate is the majority); macro-F1 over the real")
print("classes is the number for the README.")

family_support = Counter(family_labels)
print("\nreal-holdout support per class:", {FAMILY_IDS[i]: int(family_support.get(i, 0)) for i in present})

if family_devonly_raw:
    dev_ds = Dataset.from_dict({
        "text": [full_transcript(r) for r in family_devonly_raw],
        "label": [FAMILY_LABEL2ID[r["family"]] for r in family_devonly_raw],
    })
    dev_tok = dev_ds.map(tokenize_family, batched=True, remove_columns=["text"])
    dev_out = family_trainer.predict(dev_tok)
    dev_preds = np.argmax(dev_out.predictions, axis=-1)
    dev_labels = dev_out.label_ids
    dev_acc = (dev_preds == dev_labels).mean()
    dev_classes = sorted({FAMILY_IDS[i] for i in dev_labels.tolist()})
    print()
    print(f"dev-only SYNTHETIC sanity check — {dev_classes}, {len(family_devonly_raw)} rows")
    print(f"  accuracy on in-distribution synthetic: {dev_acc:.3f}")
    print("  shows the model learned these classes; NOT real-world generalization evidence.")
    print("  (no public real-derived eval exists for them — the only source mislabelled them.)")

## Calibration

Temperature scaling (Guo et al. 2017) fitted on the **validation** split, never on the locked test — fitting it on the test set would make both the accuracy and the calibration look better than they are. Stored as a scalar `T`; the serving API divides logits by `T` before softmax and reports `calibrated: true`. ECE before/after is reported on the locked test.

In [ ]:
class TemperatureScaler(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = torch.nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, logits):
        return logits / self.temperature


def fit_temperature(logits, labels):
    scaler = TemperatureScaler()
    logits_t = torch.tensor(logits, dtype=torch.float)
    labels_t = torch.tensor(labels, dtype=torch.long)
    optimizer = torch.optim.LBFGS([scaler.temperature], lr=0.01, max_iter=50)
    nll = torch.nn.CrossEntropyLoss()

    def closure():
        optimizer.zero_grad()
        loss = nll(scaler(logits_t), labels_t)
        loss.backward()
        return loss

    optimizer.step(closure)
    return scaler.temperature.item()


def expected_calibration_error(probs, labels, n_bins=10):
    conf = probs.max(axis=1)
    preds = probs.argmax(axis=1)
    correct = (preds == labels).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        ece += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
    return ece


def softmax_np(x):
    x = x - x.max(axis=-1, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=-1, keepdims=True)


family_temperature = fit_temperature(family_val_output.predictions, family_val_output.label_ids)
print(f"fitted temperature (on validation, never the locked test): {family_temperature:.4f}")

test_logits = family_eval_output.predictions
ece_before = expected_calibration_error(softmax_np(test_logits), family_labels)
ece_after = expected_calibration_error(softmax_np(test_logits / family_temperature), family_labels)
print(f"locked-test ECE before calibration: {ece_before:.4f}")
print(f"locked-test ECE after  calibration: {ece_after:.4f}")

with open(OUTPUT_DIR / "calibration.json", "w") as f:
    json.dump({
        "family_temperature": family_temperature,
        "ece_before": float(ece_before),
        "ece_after": float(ece_after),
        "fitted_on": "validation split carved from train by lineage",
    }, f, indent=2)

## Export family model to ONNX INT8

Exported immediately after training/eval/calibration — while the model is still hot in this session, not deferred to the end.

In [ ]:
import shutil

import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from transformers import AutoModelForSequenceClassification, AutoModel


class LogitsWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask):
        return self.model(input_ids=input_ids, attention_mask=attention_mask).logits


class HiddenWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask):
        return self.model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state


def export_wrapped(wrapper, tokenizer, fp32_path, sample_text, max_len):
    fp32_path.parent.mkdir(parents=True, exist_ok=True)
    dummy = tokenizer([sample_text], return_tensors="pt", truncation=True, max_length=min(64, max_len), padding=True)
    with torch.no_grad():
        torch.onnx.export(
            wrapper.cpu().eval(), (dummy["input_ids"], dummy["attention_mask"]), str(fp32_path),
            input_names=["input_ids", "attention_mask"], output_names=["output"],
            dynamic_axes={"input_ids": {0: "b", 1: "s"}, "attention_mask": {0: "b", 1: "s"}, "output": {0: "b"}},
            opset_version=17, do_constant_folding=True, dynamo=False,
        )
    print(f"  exported fp32 ONNX -> {fp32_path}")


def onnx_logits(onnx_file, tokenizer, texts, max_len, batch=8):
    sess = ort.InferenceSession(str(onnx_file), providers=["CPUExecutionProvider"])
    names = {i.name for i in sess.get_inputs()}
    chunks = []
    for i in range(0, len(texts), batch):
        enc = tokenizer(texts[i:i + batch], truncation=True, max_length=max_len, padding=True, return_tensors="np")
        feed = {k: enc[k].astype(np.int64) for k in enc if k in names}
        chunks.append(sess.run(None, feed)[0])
    return np.concatenate(chunks, axis=0)


def torch_logits(model, tokenizer, texts, max_len, batch=8):
    model.eval()
    chunks = []
    for i in range(0, len(texts), batch):
        enc = tokenizer(texts[i:i + batch], truncation=True, max_length=max_len, padding=True, return_tensors="pt")
        with torch.no_grad():
            chunks.append(model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits.cpu().numpy())
    return np.concatenate(chunks, axis=0)


def export_classifier(trainer, tokenizer, hf_dir, onnx_dir, int8_dir, serving_dir, eval_ds, max_len, sample_text):
    trainer.save_model(str(hf_dir))
    tokenizer.save_pretrained(str(hf_dir))
    cpu_model = AutoModelForSequenceClassification.from_pretrained(str(hf_dir), reference_compile=False).cpu().eval().float()

    texts = list(eval_ds["text"])
    labels = np.array(eval_ds["label"])
    present = sorted(set(labels.tolist()))
    pt_preds = np.argmax(torch_logits(cpu_model, tokenizer, texts, max_len), axis=-1)

    fp32_path = onnx_dir / "model.onnx"
    int8_path = int8_dir / "model_quantized.onnx"
    export_wrapped(LogitsWrapper(cpu_model), tokenizer, fp32_path, sample_text, max_len)

    pt_f1 = f1_score(labels, pt_preds, labels=present, average="macro", zero_division=0)
    fp32_preds = np.argmax(onnx_logits(fp32_path, tokenizer, texts, max_len), axis=-1)
    fp32_f1 = f1_score(labels, fp32_preds, labels=present, average="macro", zero_division=0)
    fp32_agree = (fp32_preds == pt_preds).mean()

    precision, src, served = "fp32", fp32_path, fp32_f1
    try:
        int8_dir.mkdir(parents=True, exist_ok=True)
        stripped = onnx.load(str(fp32_path))
        del stripped.graph.value_info[:]
        clean_path = fp32_path.with_name("model_clean.onnx")
        onnx.save(stripped, str(clean_path))
        quantize_dynamic(str(clean_path), str(int8_path), weight_type=QuantType.QInt8, per_channel=True, op_types_to_quantize=["MatMul"])
        int8_preds = np.argmax(onnx_logits(int8_path, tokenizer, texts, max_len), axis=-1)
        int8_f1 = f1_score(labels, int8_preds, labels=present, average="macro", zero_division=0)
        int8_agree = (int8_preds == pt_preds).mean()
        print(f"  PyTorch {pt_f1:.4f} | fp32 {fp32_f1:.4f} agree {fp32_agree:.4f} | INT8 {int8_f1:.4f} agree {int8_agree:.4f}")
        if int8_agree >= 0.95 and int8_f1 >= pt_f1 - 0.03:
            precision, src, served = "int8", int8_path, int8_f1
    except Exception as exc:
        print(f"  INT8 quantization failed ({type(exc).__name__}) -> serving fp32 ({fp32_f1:.4f})")

    if serving_dir.exists():
        shutil.rmtree(serving_dir)
    serving_dir.mkdir(parents=True)
    shutil.copy(src, serving_dir / "model.onnx")
    tokenizer.save_pretrained(str(serving_dir))
    json.dump({"precision": precision, "macro_f1": float(served), "pytorch_macro_f1": float(pt_f1)}, open(serving_dir / "serving.json", "w"), indent=2)
    print(f"  serving {precision}: macro-F1 {served:.4f} -> {serving_dir}")


print("family:")
export_classifier(family_trainer, family_tokenizer, OUTPUT_DIR / "family_hf", OUTPUT_DIR / "family_onnx",
                  OUTPUT_DIR / "family_int8", OUTPUT_DIR / "family_serving", family_eval_ds, FAMILY_MAX_LENGTH,
                  "namaste this is inspector kumar from mumbai police your parcel has illegal items transfer now")

In [ ]:
print("family export + quantize + parity done above (torch.onnx, no optimum).")

In [ ]:
print("family export handled without optimum (torch.onnx + onnxruntime). nothing to do here.")

## Run B — stage classifier

Utterance-level, max_length 128, 6 labels.

**Data choice.** The stage model reads one utterance at a time, so it needs clean single-turn examples, and it needs them from both speakers — production ASR segments a whole call, so it is handed victim speech too.

- **BothBosu dialogues** are the only source with properly segmented conversational turns.
- **`groq_scratch_*`** are the advisory-grounded Indian/Hinglish playbook calls (9–16 turns each). These are the *only* in-domain `digital_arrest` stage data that exists.
- **`kaggle_ieee_scam` is dropped**: its rows are multi-stage call *summaries* (one row spans authority-claim → threat → payment yet carries a single arbitrary label — ungradeable per-utterance).
- **`augment_of_*` is dropped**: every one of those 720 rows is single-turn — paraphrases of the summary blobs and of email spam, not call utterances.

The split is held out **by lineage** (`parent_dialogue_id`, falling back to `dialogue_id`), so paraphrase variants of one parent can never straddle train/eval — the same leak that faked the family metric.

Eval is therefore held-out dialogue utterances, largely synthetic — the same honesty caveat as the `digital_arrest` family eval (no public per-utterance stage-labelled real Indian dataset exists).

In [ ]:
BOTHBOSU_SOURCES = {
    "bothbosu_scam_dialogue", "bothbosu_single_agent", "bothbosu_multi_agent",
    "bothbosu_scammer_conversation",
}

stage_rows = load_jsonl(DATA_DIR / "stage_labels.jsonl")
print(f"stage rows total: {len(stage_rows)}")


def is_playbook_call(row):
    return row["source"].startswith("groq_scratch_")


usable = [r for r in stage_rows if r["source"] in BOTHBOSU_SOURCES or is_playbook_call(r)]
indian_extra = [
    r for r in stage_rows
    if r["source"] == "kaggle_fraud_call_india" and r["stage"] != "s0_none"
]

stage_keys = sorted({lineage_key(r) for r in usable})
stage_rng = random.Random(SEED)
stage_rng.shuffle(stage_keys)
stage_eval_keys = set(stage_keys[:round(len(stage_keys) * 0.18)])

stage_eval_raw = [r for r in usable if lineage_key(r) in stage_eval_keys]
stage_train_raw = [r for r in usable if lineage_key(r) not in stage_eval_keys] + indian_extra

print(f"usable utterances: {len(usable)} (summary blobs and single-turn augments excluded)")
print(f"stage train: {len(stage_train_raw)}, stage eval: {len(stage_eval_raw)}")
print(f"held-out lineages: {len(stage_eval_keys)} of {len(stage_keys)}")
print("eval support per stage:", dict(sorted(Counter(r["stage"] for r in stage_eval_raw).items())))

da_eval = [r for r in stage_eval_raw if r["family"] == "digital_arrest"]
print(f"\ndigital_arrest utterances in stage eval: {len(da_eval)}")
print("  by stage:", dict(sorted(Counter(r["stage"] for r in da_eval).items())))

victim_eval = sum(1 for r in stage_eval_raw if r.get("label_method") == "rule_victim_turn")
print(f"victim utterances in stage eval: {victim_eval} (production feeds these too)")

In [ ]:
stage_train_ds = Dataset.from_dict({
    "text": [r["text"] for r in stage_train_raw],
    "label": [STAGE_LABEL2ID[r["stage"]] for r in stage_train_raw],
})
stage_eval_ds = Dataset.from_dict({
    "text": [r["text"] for r in stage_eval_raw],
    "label": [STAGE_LABEL2ID[r["stage"]] for r in stage_eval_raw],
})

stage_counts = Counter(stage_train_ds["label"])
stage_total = sum(stage_counts.values())
_sraw = np.array([stage_total / (len(STAGE_IDS) * stage_counts.get(i, 1)) for i in range(len(STAGE_IDS))])
_sw = _sraw
stage_class_weights = torch.tensor(_sw, dtype=torch.float)
print("stage class weights (inverse-frequency):", dict(zip(STAGE_IDS, [round(float(x), 2) for x in _sw])))

In [ ]:
stage_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
STAGE_MAX_LENGTH = 128


def tokenize_stage(batch):
    return stage_tokenizer(batch["text"], truncation=True, max_length=STAGE_MAX_LENGTH)


stage_train_tok = stage_train_ds.map(tokenize_stage, batched=True, remove_columns=["text"])
stage_eval_tok = stage_eval_ds.map(tokenize_stage, batched=True, remove_columns=["text"])

stage_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(STAGE_IDS),
    id2label=STAGE_ID2LABEL,
    label2id=STAGE_LABEL2ID,
    reference_compile=False,
)

In [ ]:
stage_collator = DataCollatorWithPadding(tokenizer=stage_tokenizer)

stage_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "stage_checkpoints"),
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    num_train_epochs=6,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    dataloader_num_workers=2,
    logging_steps=25,
    report_to=[],
    seed=SEED,
)

stage_trainer = WeightedTrainer(
    model=stage_model,
    args=stage_args,
    train_dataset=stage_train_tok,
    eval_dataset=stage_eval_tok,
    data_collator=stage_collator,
    class_weights=stage_class_weights,
    compute_metrics=make_compute_metrics(STAGE_IDS),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

stage_trainer.train()

In [ ]:
stage_eval_output = stage_trainer.predict(stage_eval_tok)
stage_preds = np.argmax(stage_eval_output.predictions, axis=-1)
stage_labels_arr = stage_eval_output.label_ids

print(classification_report(stage_labels_arr, stage_preds, target_names=STAGE_IDS, zero_division=0))

stage_cm = confusion_matrix(stage_labels_arr, stage_preds, labels=list(range(len(STAGE_IDS))))
plt.figure(figsize=(8, 6))
sns.heatmap(stage_cm, annot=True, fmt="d", xticklabels=STAGE_IDS, yticklabels=STAGE_IDS, cmap="Oranges")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Stage classifier — held-out dialogue utterances (synthetic; see model card)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "stage_confusion_matrix.png", dpi=150)
plt.show()

S0_ID = STAGE_LABEL2ID["s0_none"]
benign = stage_labels_arr == S0_ID
escalated = stage_preds != S0_ID

stage_accuracy = (stage_preds == stage_labels_arr).mean()
stage_macro_f1 = f1_score(stage_labels_arr, stage_preds, average="macro", zero_division=0)
stage_trivial = benign.mean()
false_escalation = (escalated & benign).sum() / max(benign.sum(), 1)

print("=" * 62)
print(f"accuracy                        {stage_accuracy:.4f}   <- threshold {ACCURACY_THRESHOLD:.2f}")
print(f"macro-F1                        {stage_macro_f1:.4f}")
print(f"always-answer-s0_none           {stage_trivial:.4f}")
print()
print(f"false-escalation on benign      {false_escalation:.4f}   <- decides whether a")
print(f"benign support (incl. victim)   {benign.sum()}          legit call trips the gauge")
print("=" * 62)

if stage_accuracy >= ACCURACY_THRESHOLD:
    print(f"PASS — stage accuracy {stage_accuracy:.1%} clears the {ACCURACY_THRESHOLD:.0%} bar")
else:
    print(f"FAIL — stage accuracy {stage_accuracy:.1%} is under the {ACCURACY_THRESHOLD:.0%} bar")

print()
print(f"THRESHOLD SUMMARY: family {family_accuracy:.1%} | stage {stage_accuracy:.1%}")
if max(family_accuracy, stage_accuracy) >= ACCURACY_THRESHOLD:
    print(f"at least one model clears {ACCURACY_THRESHOLD:.0%}")
else:
    print(f"NEITHER model clears {ACCURACY_THRESHOLD:.0%} — see the options printed above")

In [ ]:
print("stage:")
export_classifier(stage_trainer, stage_tokenizer, OUTPUT_DIR / "stage_hf", OUTPUT_DIR / "stage_onnx",
                  OUTPUT_DIR / "stage_int8", OUTPUT_DIR / "stage_serving", stage_eval_ds, STAGE_MAX_LENGTH,
                  "sir aapko apna aadhaar aur otp abhi share karna hoga")

## Embedder + FAISS index

`intfloat/multilingual-e5-small` → ONNX INT8, then a FAISS index over every dialogue in `scripts_index.jsonl` for the closest-known-script feature. E5 convention: index passages with a `"passage: "` prefix; the serving API will prefix queries with `"query: "`.

In [ ]:
E5_MODEL = "intfloat/multilingual-e5-small"
e5_tokenizer = AutoTokenizer.from_pretrained(E5_MODEL)
e5_model = AutoModel.from_pretrained(E5_MODEL).cpu().eval().float()

e5_fp32 = OUTPUT_DIR / "e5_onnx" / "model.onnx"
e5_int8 = OUTPUT_DIR / "e5_serving" / "model.onnx"
print("e5 embedder:")
export_wrapped(HiddenWrapper(e5_model), e5_tokenizer, e5_fp32, "query: this is a test", 512)
(OUTPUT_DIR / "e5_serving").mkdir(parents=True, exist_ok=True)
quantize_dynamic(str(e5_fp32), str(e5_int8), weight_type=QuantType.QInt8, per_channel=True, op_types_to_quantize=["MatMul"])
e5_tokenizer.save_pretrained(str(OUTPUT_DIR / "e5_serving"))
print(f"  e5 INT8 -> {e5_int8}")

In [ ]:
import faiss

scripts_index_rows = load_jsonl(DATA_DIR / "scripts_index.jsonl")
print(f"scripts_index rows: {len(scripts_index_rows)}")

e5_sess = ort.InferenceSession(str(e5_int8), providers=["CPUExecutionProvider"])
e5_names = {i.name for i in e5_sess.get_inputs()}
e5_embed_tokenizer = AutoTokenizer.from_pretrained(str(OUTPUT_DIR / "e5_serving"))


def embed_passages(texts, batch_size=64):
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = ["passage: " + t[:2000] for t in texts[i:i + batch_size]]
        enc = e5_embed_tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="np")
        feed = {k: enc[k].astype(np.int64) for k in enc if k in e5_names}
        hidden = e5_sess.run(None, feed)[0]
        mask = enc["attention_mask"][:, :, None].astype(np.float32)
        pooled = (hidden * mask).sum(1) / np.clip(mask.sum(1), 1e-9, None)
        pooled = pooled / np.clip(np.linalg.norm(pooled, axis=1, keepdims=True), 1e-9, None)
        vecs.append(pooled.astype(np.float32))
    return np.concatenate(vecs, axis=0)


script_embeddings = embed_passages([r["text"] for r in scripts_index_rows])
print(script_embeddings.shape)

In [ ]:
faiss_index = faiss.IndexFlatIP(script_embeddings.shape[1])
faiss_index.add(script_embeddings.astype(np.float32))
faiss.write_index(faiss_index, str(OUTPUT_DIR / "faiss.index"))

scripts_meta = [
    {"script_id": r["script_id"], "family": r["family"], "excerpt": r["text"][:120]}
    for r in scripts_index_rows
]
with open(OUTPUT_DIR / "scripts_meta.json", "w", encoding="utf-8") as f:
    json.dump(scripts_meta, f, ensure_ascii=False, indent=2)

print(f"faiss index: {faiss_index.ntotal} vectors")

## Summary

Everything under `OUTPUT_DIR` is what the HF Space image needs: both INT8 classifiers, the INT8 embedder, the FAISS index + metadata, calibration, and both confusion matrix PNGs for the README/model card.

In [ ]:
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(OUTPUT_DIR)}  ({p.stat().st_size / 1024:.1f} KB)")